# 02 — Within-Subject Benchmark

This notebook runs the main within-subject benchmark for EEG stimulus classification on the selected OpenNeuro ERP/oddball dataset.

## Goal

Compare the following models under the same grouped evaluation design:

- **LDA**
- **Logistic Regression**
- **EEGNet**

## Evaluation design

For each selected subject:

1. reconstruct trials in chronological order
2. crop to the main benchmark time window
3. split trials into contiguous chunks
4. build **purged grouped outer folds**
5. evaluate classical models directly on the outer folds
6. evaluate EEGNet with:
   - group-aware validation inside each training fold
   - early stopping on validation AUC
   - final testing on the held-out outer fold

## Outputs

All outputs are saved under:

`results/<dataset>/<task>/<signal>/<eval_mode>/<control_mode>/`

This notebook produces:

- seed-specific fold-level result tables
- combined fold-level result tables across seeds
- subject-level summaries
- model-level summaries
- per-subject comparison plots
- model ranking plots
- representative EEGNet learning-curve figures
- experiment metadata

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

import pandas as pd

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment, get_experiment_family
from oddball_eeg.io import save_dataframe, save_experiment_metadata, save_history_dict
from oddball_eeg.loader import load_subjects_grouped, make_subject_summary_table
from oddball_eeg.preprocessing import apply_crop_to_loaded_dataset
from oddball_eeg.reproducibility import get_default_device
from oddball_eeg.splits import build_folds_by_subject
from oddball_eeg.summaries import (
    summarize_fold_results,
    summarize_model_results_across_subjects,
    summarize_subject_results_across_seeds,
)
from oddball_eeg.training import evaluate_classical_benchmark, evaluate_eegnet_benchmark
from oddball_eeg.visualization import (
    make_model_ranking_table,
    make_subject_metric_table,
    plot_learning_curve,
    plot_model_ranking_bars,
    plot_subject_metric_bars,
    plot_subject_seed_stability,
    print_model_ranking,
    print_subject_model_summary,
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)

In [ ]:
set_pandas_display_options()

## 1. Load benchmark configuration

We use the frozen main within-subject benchmark configuration.

In [ ]:
# Change only this line when switching experiments.
# Standard pooled runs:
# EXPERIMENT_NAME = "ds003061_within_subject_max15"
# EXPERIMENT_NAME = "ds005863_within_subject_max15"
# EXPERIMENT_NAME = "ds006466_within_subject_max15"
# EXPERIMENT_NAME = "ds006480_within_subject_max15"

# Time-binned refinement runs:
# EXPERIMENT_NAME = "ds003061_within_subject_max15_timebin"
# EXPERIMENT_NAME = "ds005863_within_subject_max15_timebin"
# EXPERIMENT_NAME = "ds006466_within_subject_max15_timebin"
# EXPERIMENT_NAME = "ds006480_within_subject_max15_timebin"

# Wide-window time-binned refinement runs:
# EXPERIMENT_NAME = "ds003061_within_subject_max15_wide_timebin"
# EXPERIMENT_NAME = "ds005863_within_subject_max15_wide_timebin"
# EXPERIMENT_NAME = "ds006466_within_subject_max15_wide_timebin"
# EXPERIMENT_NAME = "ds006480_within_subject_max15_wide_timebin"

# ds006480 condition-separated refinement runs:
# EXPERIMENT_NAME = "ds006480_active_within_subject_max15"
# EXPERIMENT_NAME = "ds006480_passive_within_subject_max15"
# EXPERIMENT_NAME = "ds006480_active_within_subject_max15_timebin"
# EXPERIMENT_NAME = "ds006480_passive_within_subject_max15_timebin"

# EXPERIMENT_NAME = "ds003061_within_subject_max30_wide_timebin"
# EXPERIMENT_NAME = "ds005863_within_subject_max30_wide_timebin"
# EXPERIMENT_NAME = "ds006466_within_subject_max30_wide_timebin"
# EXPERIMENT_NAME = "ds006480_within_subject_max30_wide_timebin"

EXPERIMENT_NAME = "ds006480_within_subject_max30_wide_timebin"

# ------------------------------------------------------------
# Notebook 02 is a single-experiment within-subject benchmark.
# It should load exactly one ExperimentConfig.
#
# Do NOT call get_experiment_family() here.
# Families are only needed in Notebook 03 for controls.
# ------------------------------------------------------------

cfg = get_experiment(EXPERIMENT_NAME)

cfg_perm = None
cfg_prestim = None
cfg_loso = None
has_experiment_family = False

device = get_default_device()

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "02_within_subject_benchmark.ipynb",
        "purpose": "Grouped within-subject benchmark or standalone within-subject refinement.",
        "experiment_key": EXPERIMENT_NAME,
        "has_experiment_family": has_experiment_family,
    },
)

print("Experiment key :", EXPERIMENT_NAME)
print("Experiment name:", cfg.name)
print("Has family     :", has_experiment_family)
print("Eval mode      :", cfg.eval_mode)
print("Control mode   :", cfg.control_mode)
print("Dataset        :", cfg.dataset.dataset)
print("Task           :", cfg.dataset.task)
print("Event preset   :", cfg.dataset.event_preset)
print("Run tag        :", cfg.run_tag)
print("Subjects       :", cfg.subjects.subjects)
print("Seeds          :", cfg.seeds.seeds)
print("Signal config  :", cfg.signal)
print("Feature mode   :", cfg.feature.mode)
print("Feature bin ms :", cfg.feature.bin_ms)
print("Crop window    :", cfg.effective_crop_window())
print("Chunk size     :", cfg.splits.group_chunk_size)
print("Purge neighbors:", cfg.splits.purge_neighbor_groups)
print("Device         :", device)

print("\nResult directory:")
print(cfg.result_dir())

print("\nSeed result example:")
print(cfg.result_dir(seed=cfg.seeds.seeds[0]))

print("\nSummary table directory:")
print(cfg.table_dir("summaries"))

print("\nBenchmark figure directory:")
print(cfg.figure_dir("benchmark"))

print("\nStandalone Notebook 02 run:")
print("This notebook will train/evaluate exactly this experiment config.")

## 2. Load grouped chronological subject data

In [ ]:
epoching_cfg = make_epoching_config_from_experiment(cfg)

print(epoching_cfg)

data_full = load_subjects_grouped(
    dataset=cfg.dataset.dataset,
    task_name=cfg.dataset.task,
    subjects=cfg.subjects.subjects,
    epoching_cfg=epoching_cfg,
    chunk_size=cfg.splits.group_chunk_size,
    apply_baseline=True,
    verbose=True,
)

## 3. Apply the main crop window

In [ ]:
crop_window = cfg.effective_crop_window()
assert crop_window is not None, "Expected a crop window for the main benchmark."

data_main = apply_crop_to_loaded_dataset(
    data_full,
    tmin=crop_window[0],
    tmax=crop_window[1],
)

subject_summary_main = make_subject_summary_table(data_main)
subject_summary_main = round_numeric_columns(subject_summary_main, decimals=4)

display(subject_summary_main)

save_dataframe(
    subject_summary_main,
    cfg.table_dir("summaries") / "loaded_subject_summary_main_cropped.csv",
)

## 4. Build grouped folds for each subject

These folds are the outer evaluation folds used for all models in the within-subject benchmark.

In [ ]:
folds_by_sub = build_folds_by_subject(
    data=data_main,
    n_splits=cfg.splits.n_splits,
    purge_neighbors=cfg.splits.purge_neighbor_groups,
)

for sub in data_main.subjects:
    print("=" * 70)
    print(f"Subject {sub}")
    print(f"Number of folds: {len(folds_by_sub[sub])}")

fold_summary_rows = []

for sub in data_main.subjects:
    for fold_id, (tr_idx, te_idx) in enumerate(folds_by_sub[sub], start=1):
        y = data_main.y_by_sub[sub]
        fold_summary_rows.append({
            "subject": sub,
            "fold": int(fold_id),
            "n_train": int(len(tr_idx)),
            "n_test": int(len(te_idx)),
            "train_target_rate": float(y[tr_idx].mean()),
            "test_target_rate": float(y[te_idx].mean()),
        })

df_fold_design = pd.DataFrame(fold_summary_rows)
df_fold_design = round_numeric_columns(df_fold_design, decimals=4)

display(df_fold_design)

save_dataframe(
    df_fold_design,
    cfg.table_dir("fold_design") / "outer_fold_design.csv",
)

## 5. Run benchmark across seeds

For each seed, we run:

- classical baselines on the same grouped outer folds
- EEGNet on the same grouped outer folds, with group-aware inner validation

In [ ]:
all_fold_results = []
all_eegnet_histories_by_seed = {}

for seed in cfg.seeds.seeds:
    print("#" * 90)
    print(f"RUNNING SEED {seed}")
    print("#" * 90)

    seed_fold_dir = cfg.table_dir("folds", seed=seed)
    seed_history_dir = cfg.history_dir("eegnet", seed=seed)

    # ------------------------------------------------------------
    # Classical baselines
    # ------------------------------------------------------------
    df_classical = evaluate_classical_benchmark(
        data=data_main,
        folds_by_sub=folds_by_sub,
        model_names=("lda", "logreg"),
        seed=seed,
        verbose=cfg.verbose,
        feature_mode=cfg.feature.mode,
        bin_ms=cfg.feature.bin_ms,
    )

    save_dataframe(df_classical, seed_fold_dir / "classical_fold_results.csv")

    # ------------------------------------------------------------
    # EEGNet
    # ------------------------------------------------------------
    df_eegnet, histories = evaluate_eegnet_benchmark(
        data=data_main,
        folds_by_sub=folds_by_sub,
        device=device,
        normalize=cfg.eegnet.normalize,
        batch_size=cfg.eegnet.batch_size,
        lr=cfg.eegnet.lr,
        weight_decay=cfg.eegnet.weight_decay,
        epochs=cfg.eegnet.epochs,
        patience=cfg.eegnet.early_stopping_patience,
        val_size=cfg.splits.val_size,
        val_n_tries=cfg.splits.val_n_tries,
        use_class_weight=cfg.eegnet.use_class_weight,
        grad_clip_norm=cfg.eegnet.grad_clip_norm,
        eegnet_kwargs={
            "n_classes": cfg.eegnet.n_classes,
            "f1": cfg.eegnet.f1,
            "d": cfg.eegnet.d,
            "f2": cfg.eegnet.f2,
            "k1": cfg.eegnet.k1,
            "k2": cfg.eegnet.k2,
            "dropout": cfg.eegnet.dropout,
        },
        seed=seed,
        verbose=cfg.verbose,
    )

    save_dataframe(df_eegnet, seed_fold_dir / "eegnet_fold_results.csv")

    save_history_dict(
        histories,
        seed_history_dir,
        filename_prefix="eegnet_history",
    )

    all_eegnet_histories_by_seed[seed] = histories

    # ------------------------------------------------------------
    # Combine this seed
    # ------------------------------------------------------------
    df_seed = pd.concat([df_classical, df_eegnet], ignore_index=True)
    save_dataframe(df_seed, seed_fold_dir / "all_fold_results.csv")

    all_fold_results.append(df_seed)

## 6. Combine all fold-level results across seeds

In [ ]:
df_all_folds = pd.concat(all_fold_results, ignore_index=True)

SUMMARY_DIR = cfg.table_dir("summaries")

save_dataframe(
    df_all_folds,
    SUMMARY_DIR / "all_fold_results_across_seeds.csv",
)

print("Combined fold-level shape:", df_all_folds.shape)

display(
    round_numeric_columns(df_all_folds.head(), decimals=4)
)

## 7. Summarize folds into subject-level seed summaries

For each `(subject, model, seed)`, we compute:

- mean/std ROC-AUC
- mean/std PR-AUC
- mean/std balanced accuracy
- mean/std accuracy
- mean/std F1

In [ ]:
df_subject_seed_summary = summarize_fold_results(
    df_all_folds,
    group_cols=("subject", "model", "seed"),
)

save_dataframe(
    df_subject_seed_summary,
    SUMMARY_DIR / "subject_seed_summary.csv",
)

display(
    round_numeric_columns(df_subject_seed_summary, decimals=4)
)

## 8. Aggregate subject summaries across seeds

This gives one row per `(subject, model)` and measures seed stability.

In [ ]:
df_subject_across_seeds = summarize_subject_results_across_seeds(df_subject_seed_summary)

save_dataframe(
    df_subject_across_seeds,
    SUMMARY_DIR / "subject_summary_across_seeds.csv",
)

display(
    round_numeric_columns(df_subject_across_seeds, decimals=4)
)

## 9. Aggregate to model-level rankings across subjects

This is the main model comparison table for the within-subject benchmark.

In [ ]:
df_model_summary = summarize_model_results_across_subjects(df_subject_across_seeds)

save_dataframe(
    df_model_summary,
    SUMMARY_DIR / "model_summary_across_subjects.csv",
)

display(
    round_numeric_columns(df_model_summary, decimals=4)
)

## 10. Subject-wise mean ROC-AUC table

In [ ]:
table_subject_auc = make_subject_metric_table(
    df_subject_across_seeds.rename(columns={"auc_seed_mean": "auc_mean"}),
    metric_col="auc_mean",
)
display(table_subject_auc)

save_dataframe(
    table_subject_auc,
    cfg.table_dir("reporting") / "subject_auc_table.csv",
)

## 11. Subject-wise seed stability table

In [ ]:
table_seed_stability = round_numeric_columns(df_subject_across_seeds, decimals=4)
display(table_seed_stability)

save_dataframe(
    table_seed_stability,
    cfg.table_dir("reporting") / "seed_stability_table.csv",
)

## 12. Model ranking table

In [ ]:
table_model_ranking = make_model_ranking_table(
    df_model_summary,
    metric_col="mean_subject_auc",
)
display(table_model_ranking)

save_dataframe(
    table_model_ranking,
    cfg.table_dir("reporting") / "model_ranking_table.csv",
)

In [ ]:
print_subject_model_summary(
    df_subject_across_seeds.rename(
        columns={
            "auc_seed_mean": "auc_mean",
            "auc_seed_std": "auc_std",
        }
    ),
    metric_mean_col="auc_mean",
    metric_std_col="auc_std",
)

In [ ]:
print_model_ranking(
    df_model_summary,
    metric_col="mean_subject_auc",
)

In [ ]:
fig_auc = plot_subject_metric_bars(
    df_subject_across_seeds.rename(
        columns={
            "auc_seed_mean": "auc_mean",
            "auc_seed_std": "auc_std",
        }
    ),
    metric_col="auc_mean",
    error_col="auc_std",
    title=f"Within-subject benchmark: ROC-AUC by subject — {cfg.dataset.dataset}, {cfg.dataset.task}",
    ylabel="ROC-AUC",
)
fig_auc

## 14. Per-subject PR-AUC comparison plot

In [ ]:
fig_pr = plot_subject_metric_bars(
    df_subject_across_seeds.rename(
        columns={
            "pr_auc_seed_mean": "auc_mean",
            "pr_auc_seed_std": "auc_std",
        }
    ),
    metric_col="auc_mean",
    error_col="auc_std",
    title=f"Within-subject benchmark: PR-AUC by subject — {cfg.dataset.dataset}, {cfg.dataset.task}",
    ylabel="PR-AUC",
)
fig_pr

## 15. Subject-wise seed stability plot

In [ ]:
fig_seed = plot_subject_seed_stability(
    df_subject_across_seeds,
    metric_col="auc_seed_mean",
    error_col="auc_seed_std",
    title=f"Within-subject benchmark: seed stability — {cfg.dataset.dataset}, {cfg.dataset.task}",
    ylabel="ROC-AUC across seeds",
)
fig_seed

## 16. Model ranking plot

In [ ]:
fig_rank = plot_model_ranking_bars(
    df_model_summary,
    metric_col="mean_subject_auc",
    title=f"Model ranking by mean subject ROC-AUC — {cfg.dataset.dataset}, {cfg.dataset.task}",
    ylabel="Mean subject ROC-AUC",
)
fig_rank

## 17. Learning-curve inspection for one EEGNet run

We inspect one representative learning curve to verify training behavior qualitatively.

In [ ]:
example_seed = cfg.seeds.seeds[0]
example_subject = data_main.subjects[0]
example_fold = list(all_eegnet_histories_by_seed[example_seed][example_subject].keys())[0]

hist_example = all_eegnet_histories_by_seed[example_seed][example_subject][example_fold]
hist_example.head()

In [ ]:
fig_hist = plot_learning_curve(
    hist_example,
    title=f"EEGNet learning curve — seed {example_seed}, subject {example_subject}, fold {example_fold}",
)
fig_hist

## 18. Save key tables and figures for later notebooks / thesis reporting

In [ ]:
BENCHMARK_FIG_DIR = cfg.figure_dir("benchmark")
LEARNING_CURVE_FIG_DIR = cfg.figure_dir("learning_curves")

save_figure(
    fig_auc,
    BENCHMARK_FIG_DIR / "auc_by_subject.png",
)

save_figure(
    fig_pr,
    BENCHMARK_FIG_DIR / "pr_auc_by_subject.png",
)

save_figure(
    fig_seed,
    BENCHMARK_FIG_DIR / "seed_stability.png",
)

save_figure(
    fig_rank,
    BENCHMARK_FIG_DIR / "model_ranking.png",
)

save_figure(
    fig_hist,
    LEARNING_CURVE_FIG_DIR / "example_eegnet_learning_curve.png",
)

save_experiment_metadata(
    cfg,
    extra={
        "notebook": "02_within_subject_benchmark.ipynb",
        "status": "completed",
        "experiment_key": EXPERIMENT_NAME,
        "n_subjects": int(len(data_main.subjects)),
        "n_seeds": int(len(cfg.seeds.seeds)),
        "models": sorted(df_all_folds["model"].unique().tolist()),
        "n_fold_rows": int(len(df_all_folds)),
        "summary_dir": str(SUMMARY_DIR),
        "benchmark_figure_dir": str(BENCHMARK_FIG_DIR),
        "learning_curve_figure_dir": str(LEARNING_CURVE_FIG_DIR),
    },
)

print("Saved summary tables to:", SUMMARY_DIR)
print("Saved benchmark figures to:", BENCHMARK_FIG_DIR)
print("Saved learning-curve figures to:", LEARNING_CURVE_FIG_DIR)

## 19. Benchmark conclusions

At this point, this notebook should provide:

- a fair within-subject grouped benchmark
- fold-level outputs for every model
- subject-level summaries
- model-level rankings
- seed stability measurements
- representative learning-curve inspection

These results form the **primary model-selection evidence** for the thesis.

The next notebook should evaluate **anti-overfitting controls**:
- label permutation
- pre-stimulus-only